# 🦆 Simulación del Juego Duck Hunt

**Curso:** Programación 101 — Maestría en Ciencia de Datos e IA (UTEC)
**Profesor:** Royer Rojas Malasquez

Este notebook implementa una simulación simplificada de *Duck Hunt* usando **NumPy** (matrices, aleatoriedad), **Pandas** (registro y análisis de disparos), **Matplotlib** (visualización del tablero) y **Seaborn** (gráficos estadísticos finales).

## Índice
1. Configuración inicial y constantes
2. Pantalla de configuración estilo NES (nombre del jugador, tamaño de grid, número de disparos)
3. Carga y preprocesamiento del fondo
4. Función `pato()`
5. Función `pistola()`
6. Validación de impacto (WINNER / GAME OVER + sonido)
7. Bucle principal del juego
8. Registro de resultados (CSV acumulado por jugador)
9. Visualización estadística final



## 1. Configuración inicial y constantes

Importamos las librerías obligatorias del proyecto y definimos las rutas hacia los recursos gráficos (`assets/`). Como el notebook vive dentro de `FinalProjectP101/`, las rutas son relativas a esa carpeta.

También definimos aquí la ruta del **CSV de registros históricos** (`registros_jugadores.csv`), donde se acumulan las partidas de todos los jugadores entre distintas ejecuciones del notebook.

In [9]:
# ============================================================
# LIBRERÍAS OBLIGATORIAS DEL PROYECTO
# ============================================================
import numpy as np                     # Matrices, cuadrículas y números aleatorios
import pandas as pd                     # Registro y análisis estadístico de disparos
import matplotlib.pyplot as plt         # Visualización del tablero e imágenes del juego
import seaborn as sns                   # Gráficos estadísticos finales

# ============================================================
# LIBRERÍAS DE APOYO (manejo de imágenes, UI interactiva, audio)
# ============================================================
from PIL import Image                   # Carga y manipulación de imágenes
from pathlib import Path                # Manejo de rutas de archivo multiplataforma
from IPython.display import display, HTML, Audio, clear_output

# ============================================================
# RUTAS DE RECURSOS (assets)
# ============================================================
# El notebook vive dentro de FinalProjectP101/, así que las rutas son relativas a esa carpeta.
RUTA_ASSETS = Path("assets")
RUTA_FONDO = RUTA_ASSETS / "paisaje-limpio (2).png"
RUTA_PATO = RUTA_ASSETS / "pato (1).png"
RUTA_GANADOR = RUTA_ASSETS / "ganador.jpeg"
RUTA_GAMEOVER = RUTA_ASSETS / "gameover.jpg"
RUTA_CSV_REGISTROS = Path("registros_jugadores.csv")


def verificar_assets():
    """Valida que todos los recursos gráficos necesarios existan antes de continuar."""
    rutas = [RUTA_FONDO, RUTA_PATO, RUTA_GANADOR, RUTA_GAMEOVER]
    faltantes = [str(r) for r in rutas if not r.exists()]
    if faltantes:
        raise FileNotFoundError(
            "Faltan los siguientes recursos en 'assets/': " + ", ".join(faltantes)
        )
    print("✅ Todos los recursos gráficos fueron encontrados correctamente.")


verificar_assets()

# ============================================================
# VARIABLES GLOBALES DE CONFIGURACIÓN DE PARTIDA
# ============================================================
# Se completan en la Sección 2 (pantalla estilo NES) y las usa el resto del notebook.
NOMBRE_JUGADOR = None
TAM_GRID = None
NUM_DISPAROS = None

✅ Todos los recursos gráficos fueron encontrados correctamente.


## 2. Configuración de la partida

Se piden **3 datos** al jugador, uno a la vez, con `input()`:

0. **Nombre** — solo letras, sin espacios ni símbolos.
1. **Número de disparos (n)** — por defecto 20, pero configurable (ejemplo: 10, 30, 50).
2. **Tamaño del grid (n×n)** — por defecto 4, configurable (3×3, 4×4, 5×5, ...).

### Por qué `input()` y no botones

La primera versión de esta pantalla usaba botones de `ipywidgets` (estilo consola NES). Se cambió a `input()` porque `ipywidgets` necesita que el entorno donde corre el notebook (Jupyter Lab, Jupyter clásico, VS Code, Colab...) tenga el renderizador de widgets correctamente instalado y habilitado -- si no, los botones no responden o se ven pero no hacen nada, y una vez confirmada la configuración quedaban bloqueados sin forma de volver a cambiarlos sin reiniciar el kernel.

`input()` es una función estándar de Python: funciona igual en **cualquier** notebook sin depender de ninguna extensión, y cada valor se puede volver a pedir tantas veces como haga falta -- nunca queda "trabado".

### Validación con reintento (buenas prácticas)

Cada pregunta se hace dentro de un bucle `while True` que solo termina cuando la respuesta es válida: si el jugador escribe algo incorrecto, se le explica por qué y se le vuelve a preguntar, en vez de que el programa truene o siga adelante con un valor inválido.

**Ejemplo:** si se pide el número de disparos (rango 5-99) y el jugador escribe `"abc"` o `"200"`, el programa responde `"Debe ser un número entero entre 5 y 99. Intenta de nuevo."` y vuelve a preguntar -- las celdas siguientes solo se ejecutan una vez que los 3 valores son válidos.

> Ejecuta la siguiente celda de código y respondé las 3 preguntas en el cuadro de texto que aparece arriba del notebook (o al pie, según tu entorno).

In [10]:
# ============================================================
# BLOQUE A — CONSTANTES DE CONFIGURACIÓN
# ============================================================
GRID_MIN, GRID_MAX = 3, 8            # Límites del tamaño de grid (n x n)
DISPAROS_MIN, DISPAROS_MAX = 5, 99   # Límites del número de disparos
DISPAROS_POR_DEFECTO = 20
GRID_POR_DEFECTO = 4


# ============================================================
# BLOQUE B — VALIDACIONES PURAS (fáciles de probar sin pedir input real)
# ============================================================
def validar_nombre(texto):
    """Un nombre válido tiene solo letras (sin espacios, números ni símbolos) y no está vacío."""
    return texto.isalpha()


def validar_entero_en_rango(texto, minimo, maximo):
    """
    Verifica que 'texto' represente un entero dentro de [minimo, maximo].
    Devuelve (es_valido, valor) -- valor es None si no es válido.
    """
    if not texto.isdigit():
        return False, None
    valor = int(texto)
    if not (minimo <= valor <= maximo):
        return False, None
    return True, valor

In [ ]:
# ============================================================
# BLOQUE C — LECTURA INTERACTIVA (input() con reintento)
# ============================================================
def leer_nombre_jugador():
    """Pide el nombre hasta que sea válido: solo letras, sin espacios ni símbolos."""
    while True:
        entrada = input("Nombre del jugador (solo letras, sin espacios): ").strip().upper()
        if validar_nombre(entrada):
            return entrada
        print("  -> Nombre inválido: usa solo letras, sin espacios ni números. Intenta de nuevo.")


def leer_entero_en_rango(mensaje, minimo, maximo, valor_por_defecto=None):
    """
    Pide un entero dentro de [minimo, maximo], repitiendo hasta que sea válido.
    Si valor_por_defecto no es None, dejar la respuesta en blanco (Enter) lo usa directamente.
    """
    sufijo_default = f", Enter = {valor_por_defecto}" if valor_por_defecto is not None else ""
    while True:
        entrada = input(f"{mensaje} ({minimo}-{maximo}{sufijo_default}): ").strip()
        if entrada == "" and valor_por_defecto is not None:
            return valor_por_defecto
        es_valido, valor = validar_entero_en_rango(entrada, minimo, maximo)
        if es_valido:
            return valor
        print(f"  -> Debe ser un número entero entre {minimo} y {maximo}. Intenta de nuevo.")

In [12]:
# ============================================================
# BLOQUE D — ORQUESTACIÓN: pedir los 3 datos de configuración
# ============================================================
print("=== CONFIGURACION DE PARTIDA ===")

NOMBRE_JUGADOR = leer_nombre_jugador()
NUM_DISPAROS = leer_entero_en_rango(
    "Numero de disparos (n)", DISPAROS_MIN, DISPAROS_MAX, valor_por_defecto=DISPAROS_POR_DEFECTO
)
TAM_GRID = leer_entero_en_rango(
    "Tamano del grid (n x n)", GRID_MIN, GRID_MAX, valor_por_defecto=GRID_POR_DEFECTO
)

print(f"\nListo -- Jugador: {NOMBRE_JUGADOR} | Grid: {TAM_GRID}x{TAM_GRID} | Disparos: {NUM_DISPAROS}")

## 3. Carga y preprocesamiento del fondo

Cargamos `paisaje-limpio (2).png` como una matriz NumPy y generamos **tres versiones**: color original, escala de grises y blanco/negro — tal como pide el enunciado. Luego dividimos el tablero en una cuadrícula `TAM_GRID x TAM_GRID` calculada dinámicamente.

### 3.1 Escala de grises: fórmula manual (explicación con ejemplo)

Una imagen a color es una matriz 3D `(alto, ancho, 3)` — el último eje son los canales **R, G, B**. Para convertirla a un solo canal de grises usamos la **fórmula de luminosidad**, que pondera cada canal según cuánto brillo percibe el ojo humano:

```
gris = 0.299·R + 0.587·G + 0.114·B
```

**Ejemplo numérico:** un píxel con `R=200, G=150, B=100`:

```
gris = 0.299×200 + 0.587×150 + 0.114×100
     = 59.8 + 88.05 + 11.4
     = 159.25  →  redondeado: 159
```

Con NumPy esto se aplica a **toda la imagen a la vez** (sin bucles) usando `np.dot(imagen[...,:3], [0.299, 0.587, 0.114])` — una multiplicación matricial que reduce el eje de 3 canales a 1.

### 3.2 Blanco y negro: binarización por umbral

Sobre la matriz de grises aplicamos un **umbral** (`umbral=128` por defecto): todo píxel con valor ≥ umbral se vuelve blanco (255), y el resto negro (0).

**Mismo ejemplo:** `gris=159` con `umbral=128` → `159 ≥ 128` → el píxel se pinta **blanco (255)**.

### 3.3 División en grid (explicación con ejemplo)

Para dividir una imagen de `ancho=1920` px en `TAM_GRID=4` columnas, calculamos los **bordes** de cada celda con `np.linspace(0, ancho, columnas+1)`:

```
bordes_columna = [0, 480, 960, 1440, 1920]
```

Esto define 4 columnas: `[0-480)`, `[480-960)`, `[960-1440)`, `[1440-1920)`. La celda `(fila=i, columna=j)` del grid ocupa exactamente esa región de píxeles en la imagen original — así conectamos la matriz lógica del juego (índices `i,j`) con la matriz de píxeles real.

In [5]:
# ============================================================
# BLOQUE A — CARGA Y CONVERSIÓN DE IMAGEN (lógica pura, sin dibujar nada)
# ============================================================
PESOS_LUMINOSIDAD = np.array([0.299, 0.587, 0.114])  # pesos R, G, B
UMBRAL_BN = 128  # umbral por defecto para blanco y negro


def cargar_fondo(ruta):
    """
    Carga una imagen desde disco y la devuelve como matriz NumPy RGB (alto, ancho, 3).
    Si la imagen tiene canal alfa (RGBA), se compone sobre un fondo blanco.
    """
    imagen = Image.open(ruta)

    if imagen.mode == "RGBA":
        fondo_blanco = Image.new("RGB", imagen.size, (255, 255, 255))
        fondo_blanco.paste(imagen, mask=imagen.split()[3])  # usa el canal alfa como máscara
        imagen = fondo_blanco
    else:
        imagen = imagen.convert("RGB")

    return np.array(imagen)


def convertir_a_grises(imagen_rgb):
    """
    Convierte una matriz RGB (alto, ancho, 3) a escala de grises (alto, ancho)
    aplicando la fórmula de luminosidad: 0.299 R + 0.587 G + 0.114 B
    """
    return np.dot(imagen_rgb[..., :3], PESOS_LUMINOSIDAD).astype(np.uint8)


def convertir_a_bn(imagen_grises, umbral=UMBRAL_BN):
    """
    Binariza una matriz en escala de grises: píxeles >= umbral -> blanco (255),
    el resto -> negro (0). Devuelve una matriz del mismo tamaño en uint8.
    """
    return np.where(imagen_grises >= umbral, 255, 0).astype(np.uint8)

In [6]:
# ============================================================
# BLOQUE B — ESTRUCTURA DEL GRID (matriz de bordes de celdas)
# ============================================================
def construir_bordes_grid(alto, ancho, filas, columnas):
    """
    Calcula los bordes en píxeles de cada celda de la cuadrícula.

    Devuelve dos arreglos NumPy:
      bordes_fila    -> shape (filas+1,)    posiciones verticales de los bordes
      bordes_columna -> shape (columnas+1,) posiciones horizontales de los bordes

    Ejemplo: alto=678, ancho=1920, filas=4, columnas=4
      bordes_columna = [0, 480, 960, 1440, 1920]
      bordes_fila    = [0, 169, 339, 508, 678]
    La celda (i, j) ocupa imagen[bordes_fila[i]:bordes_fila[i+1], bordes_columna[j]:bordes_columna[j+1]]
    """
    bordes_fila = np.linspace(0, alto, filas + 1, dtype=int)
    bordes_columna = np.linspace(0, ancho, columnas + 1, dtype=int)
    return bordes_fila, bordes_columna


def centro_celda(i, j, bordes_fila, bordes_columna):
    """Devuelve las coordenadas (x, y) en píxeles del centro de la celda (i, j), útil para dibujar sprites."""
    y = (bordes_fila[i] + bordes_fila[i + 1]) // 2
    x = (bordes_columna[j] + bordes_columna[j + 1]) // 2
    return x, y


def validar_posicion_celda(i, j, filas, columnas):
    """Verifica que (i, j) sea una posición válida dentro del grid filas x columnas."""
    return 0 <= i < filas and 0 <= j < columnas

In [7]:
# ============================================================
# BLOQUE C — VISUALIZACIÓN DEL TABLERO (Matplotlib)
# ============================================================
def dibujar_tablero(ax, imagen, filas, columnas, titulo, cmap=None):
    """
    Dibuja 'imagen' en el eje 'ax' con las líneas de la cuadrícula y el número de cada celda superpuesto.
    Funciona tanto para imágenes a color (cmap=None) como en escala de grises / blanco y negro (cmap='gray').
    """
    alto, ancho = imagen.shape[0], imagen.shape[1]
    bordes_fila, bordes_columna = construir_bordes_grid(alto, ancho, filas, columnas)

    ax.imshow(imagen, cmap=cmap, vmin=0, vmax=255)
    ax.set_title(titulo, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

    # Líneas de la cuadrícula
    for borde_y in bordes_fila:
        ax.axhline(borde_y, color="red", linewidth=1)
    for borde_x in bordes_columna:
        ax.axvline(borde_x, color="red", linewidth=1)

    # Número de celda (fila, columna) en el centro de cada casilla
    for i in range(filas):
        for j in range(columnas):
            x, y = centro_celda(i, j, bordes_fila, bordes_columna)
            ax.text(x, y, f"{i},{j}", color="yellow", fontsize=8,
                     ha="center", va="center", weight="bold")


def mostrar_tres_versiones(imagen_color, imagen_grises, imagen_bn, filas, columnas):
    """Muestra lado a lado: color original, escala de grises y blanco/negro, cada una con su grid."""
    fig, ejes = plt.subplots(1, 3, figsize=(16, 5))
    dibujar_tablero(ejes[0], imagen_color, filas, columnas, "Color original")
    dibujar_tablero(ejes[1], imagen_grises, filas, columnas, "Escala de grises", cmap="gray")
    dibujar_tablero(ejes[2], imagen_bn, filas, columnas, "Blanco y negro", cmap="gray")
    plt.tight_layout()
    plt.show()

In [8]:
# ============================================================
# BLOQUE D — ORQUESTACIÓN: cargar, convertir y mostrar el tablero
# ============================================================
# Validación: la Sección 2 debe haberse confirmado con START antes de continuar.
if TAM_GRID is None:
    raise RuntimeError(
        "TAM_GRID no está definido. Vuelve a la Sección 2, configura la partida "
        "en la pantalla NES y presiona START antes de ejecutar esta celda."
    )

FONDO_COLOR = cargar_fondo(RUTA_FONDO)
FONDO_GRISES = convertir_a_grises(FONDO_COLOR)
FONDO_BN = convertir_a_bn(FONDO_GRISES)

# El enunciado pide mostrar el fondo en blanco/negro o escala de grises (no menciona
# color), asi que el tablero OPERATIVO del juego (el que usan pato(), pistola(), etc.)
# es la version en grises, expandida a 3 canales (R=G=B) para que los sprites de color
# (pato, splash) resalten sobre un fondo acromatico -- el color queda reservado a los
# sprites, no al fondo.
FONDO_JUEGO = np.stack([FONDO_GRISES] * 3, axis=-1)

ALTO_IMG, ANCHO_IMG = FONDO_COLOR.shape[0], FONDO_COLOR.shape[1]
BORDES_FILA, BORDES_COLUMNA = construir_bordes_grid(ALTO_IMG, ANCHO_IMG, TAM_GRID, TAM_GRID)

print(f"Fondo cargado: {ANCHO_IMG}x{ALTO_IMG} px  |  Grid: {TAM_GRID}x{TAM_GRID}")
mostrar_tres_versiones(FONDO_COLOR, FONDO_GRISES, FONDO_BN, TAM_GRID, TAM_GRID)

RuntimeError: TAM_GRID no está definido. Vuelve a la Sección 2, configura la partida en la pantalla NES y presiona START antes de ejecutar esta celda.

## 4. Función `pato()`

`pato()` debe: (1) elegir una celda al azar con NumPy, y (2) dibujar el sprite del pato sobre esa celda del tablero. Cada llamada reubica al pato dinámicamente — no hay ninguna opción de configuración para esto, es aleatorio en cada ronda.

### 4.1 Problema detectado: `pato (1).png` no tiene transparencia real

Al inspeccionar el archivo con NumPy (`np.unique(arr[...,3])`) se descubrió que el canal alfa es **255 en todos los píxeles** — es decir, aunque el archivo es RGBA, no hay ningún píxel transparente. Lo que hay es un **fondo celeste sólido y uniforme** `(99, 173, 255)` (el mismo color exacto en las 4 esquinas de la imagen) "horneado" como parte del dibujo.

**Solución: chroma-key.** Es la misma técnica de las pantallas verdes/azules de TV: detectamos el color de fondo y volvemos transparente todo píxel *suficientemente parecido* a ese color, midiendo la distancia euclidiana entre colores:

```
distancia = √((R - R_fondo)² + (G - G_fondo)² + (B - B_fondo)²)
```

Si `distancia ≤ tolerancia`, ese píxel se vuelve transparente (`alpha = 0`); si no, se conserva tal cual. Esto se aplica **antes** de redimensionar el sprite, para que el propio `resize` (que sí respeta el canal alfa) genere bordes suavizados alrededor del pato.

### 4.2 Mezcla alfa (composición de imágenes): explicación con ejemplo

Una vez que el sprite tiene transparencia real, lo "pegamos" sobre el fondo mezclando cada píxel proporcionalmente a su alfa:

```
resultado = alpha·sprite_rgb + (1-alpha)·fondo_rgb        (alpha entre 0 y 1)
```

**Ejemplo numérico:** un píxel del borde del pato con `alpha = 200/255 ≈ 0.784`, color del sprite `(255, 220, 0)` (amarillo pico), y el fondo en ese punto es cielo celeste `(135, 206, 235)`:

```
R = 0.784×255 + 0.216×135 ≈ 229
G = 0.784×220 + 0.216×206 ≈ 217
B = 0.784×0   + 0.216×235 ≈  51
```

Resultado: `(229, 217, 51)` — un amarillo mezclado suavemente con el celeste del fondo, en vez de un borde duro. Con NumPy esto se calcula para **todos los píxeles de la región a la vez** (sin bucles), usando *broadcasting*: `alpha` tiene shape `(alto, ancho, 1)` y se multiplica automáticamente contra los 3 canales de color.

### 4.3 Reutilización: `generar_posicion_aleatoria()`

Como tanto `pato()` como `pistola()` (Sección 5) necesitan "elegir una celda al azar dentro del grid", extraemos esa lógica a una sola función compartida — evita repetir el mismo `np.random.randint` dos veces.

In [ ]:
# ============================================================
# BLOQUE A — ALEATORIEDAD COMPARTIDA (pato y pistola la usan)
# ============================================================
def generar_posicion_aleatoria(filas, columnas):
    """Elige una celda (fila, columna) al azar dentro del grid, usando NumPy."""
    fila = np.random.randint(0, filas)
    columna = np.random.randint(0, columnas)
    return int(fila), int(columna)


# ============================================================
# BLOQUE B — COMPOSICIÓN DE IMÁGENES (chroma-key + mezcla alfa)
# ============================================================
def quitar_fondo_solido(imagen_rgba, tolerancia=40):
    """
    Vuelve transparente (alpha=0) todo píxel cuyo color esté a una distancia <= tolerancia
    del color de fondo, tomado de la esquina superior izquierda de la imagen (chroma-key).
    """
    imagen = imagen_rgba.copy()
    color_fondo = imagen[0, 0, :3].astype(float)

    diferencia = imagen[..., :3].astype(float) - color_fondo
    distancia = np.sqrt(np.sum(diferencia ** 2, axis=-1))

    imagen[..., 3] = np.where(distancia <= tolerancia, 0, imagen[..., 3])
    return imagen


def cargar_sprite(ruta, tam_px, quitar_fondo=True, tolerancia_fondo=40):
    """
    Carga un sprite, opcionalmente le quita un fondo sólido (chroma-key) y lo
    redimensiona a un cuadrado tam_px x tam_px conservando el canal alfa.
    """
    imagen = Image.open(ruta).convert("RGBA")
    arreglo = np.array(imagen)

    if quitar_fondo:
        arreglo = quitar_fondo_solido(arreglo, tolerancia_fondo)

    imagen_procesada = Image.fromarray(arreglo)
    imagen_redimensionada = imagen_procesada.resize((tam_px, tam_px), Image.LANCZOS)
    return np.array(imagen_redimensionada)


def calcular_tam_sprite(bordes_fila, bordes_columna, proporcion=0.7):
    """Calcula el tamaño (en píxeles) del sprite como una fracción del lado más chico de una celda."""
    alto_celda = bordes_fila[1] - bordes_fila[0]
    ancho_celda = bordes_columna[1] - bordes_columna[0]
    return int(min(alto_celda, ancho_celda) * proporcion)


def fusionar_sprite_en_fondo(fondo_rgb, sprite_rgba, x_centro, y_centro):
    """
    Superpone 'sprite_rgba' (con transparencia) sobre una copia de 'fondo_rgb',
    centrado en (x_centro, y_centro), usando mezcla alfa píxel a píxel.
    Si el sprite se sale de los bordes de la imagen, se recorta automáticamente.
    """
    resultado = fondo_rgb.copy()
    alto_img, ancho_img = resultado.shape[:2]
    tam = sprite_rgba.shape[0]

    x0, y0 = x_centro - tam // 2, y_centro - tam // 2
    x1, y1 = x0 + tam, y0 + tam

    # Recorte si el sprite se sale de la imagen (evita índices negativos o fuera de rango)
    recorte_x0 = max(0, -x0)
    recorte_y0 = max(0, -y0)
    recorte_x1 = tam - max(0, x1 - ancho_img)
    recorte_y1 = tam - max(0, y1 - alto_img)
    x0, y0 = max(0, x0), max(0, y0)
    x1, y1 = min(x1, ancho_img), min(y1, alto_img)

    region_fondo = resultado[y0:y1, x0:x1].astype(float)
    region_sprite = sprite_rgba[recorte_y0:recorte_y1, recorte_x0:recorte_x1].astype(float)

    alpha = region_sprite[..., 3:4] / 255.0  # shape (h, w, 1) -> broadcasting sobre los 3 canales
    mezcla = alpha * region_sprite[..., :3] + (1 - alpha) * region_fondo

    resultado[y0:y1, x0:x1] = mezcla.astype(np.uint8)
    return resultado

In [ ]:
# ============================================================
# BLOQUE C — PREPARACIÓN DEL SPRITE (se hace una sola vez, no en cada ronda)
# ============================================================
TAM_SPRITE = calcular_tam_sprite(BORDES_FILA, BORDES_COLUMNA)
SPRITE_PATO = cargar_sprite(RUTA_PATO, TAM_SPRITE)
print(f"Sprite del pato preparado: {TAM_SPRITE}x{TAM_SPRITE} px (celda: "
      f"{BORDES_COLUMNA[1]-BORDES_COLUMNA[0]}x{BORDES_FILA[1]-BORDES_FILA[0]} px)")


# ============================================================
# BLOQUE D — FUNCIÓN pato()
# ============================================================
def pato():
    """
    Genera una posición aleatoria dentro del grid (NumPy) y dibuja el pato en esa celda,
    sobre una copia del tablero (escala de grises). Devuelve la posición elegida (fila, columna).
    """
    fila, columna = generar_posicion_aleatoria(TAM_GRID, TAM_GRID)
    x_centro, y_centro = centro_celda(fila, columna, BORDES_FILA, BORDES_COLUMNA)

    tablero_con_pato = fusionar_sprite_en_fondo(FONDO_JUEGO, SPRITE_PATO, x_centro, y_centro)

    fig, ax = plt.subplots(figsize=(6, 6 * ALTO_IMG / ANCHO_IMG))
    dibujar_tablero(ax, tablero_con_pato, TAM_GRID, TAM_GRID, f"¡Apareció un pato! -> celda ({fila},{columna})")
    plt.show()

    return fila, columna


# --- Prueba de la función pato() ---
posicion_pato_demo = pato()
print("Posición del pato (demo):", posicion_pato_demo)

## 5. Función `pistola()`

`pistola()` debe: (1) elegir una celda de impacto al azar con NumPy (reutilizando `generar_posicion_aleatoria()`, ya definida en la Sección 4), y (2) mostrar visualmente dónde impactó el disparo.

### 5.1 Diferencia clave con `pato()`: vectores vs. píxeles

`pato()` manipula la **matriz de píxeles** (chroma-key + mezcla alfa) porque necesita pegar una fotografía (el sprite) sobre otra. La mira del disparo, en cambio, es una figura geométrica simple (una cruz dentro de un círculo) — no hace falta tocar ni un solo píxel de la imagen: Matplotlib puede dibujarla directamente como **líneas vectoriales** sobre los mismos ejes (`ax.plot`) donde ya se muestra el tablero. Es más simple y más nítido (no se pixela al hacer zoom).

**Ejemplo:** para dibujar el círculo de la mira con radio `r` centrado en `(x, y)`, generamos 100 ángulos equiespaciados entre 0 y 2π con `np.linspace(0, 2*np.pi, 100)` y calculamos:

```
punto_x = x + r·cos(ángulo)
punto_y = y + r·sin(ángulo)
```

Esto traza el contorno de una circunferencia — otra aplicación de NumPy (esta vez para geometría, no para imágenes).

In [ ]:
# ============================================================
# BLOQUE A — DIBUJO DE LA MIRA (geometría vectorial, no manipula píxeles)
# ============================================================
def dibujar_mira(ax, x_centro, y_centro, tam, color="red"):
    """Dibuja una mira (círculo + cruz) centrada en (x_centro, y_centro) sobre el eje 'ax'."""
    radio = tam / 2

    angulos = np.linspace(0, 2 * np.pi, 100)
    circulo_x = x_centro + radio * np.cos(angulos)
    circulo_y = y_centro + radio * np.sin(angulos)
    ax.plot(circulo_x, circulo_y, color=color, linewidth=2)

    ax.plot([x_centro - radio, x_centro + radio], [y_centro, y_centro], color=color, linewidth=2)
    ax.plot([x_centro, x_centro], [y_centro - radio, y_centro + radio], color=color, linewidth=2)


# ============================================================
# BLOQUE B — FUNCIÓN pistola()
# ============================================================
def pistola():
    """
    Genera una posición de impacto aleatoria dentro del grid (NumPy) y dibuja
    una mira sobre el tablero en esa celda. Devuelve la posición (fila, columna).
    """
    fila, columna = generar_posicion_aleatoria(TAM_GRID, TAM_GRID)
    x_centro, y_centro = centro_celda(fila, columna, BORDES_FILA, BORDES_COLUMNA)

    fig, ax = plt.subplots(figsize=(6, 6 * ALTO_IMG / ANCHO_IMG))
    dibujar_tablero(ax, FONDO_JUEGO, TAM_GRID, TAM_GRID, f"¡Disparo! -> celda ({fila},{columna})")
    dibujar_mira(ax, x_centro, y_centro, tam=TAM_SPRITE * 0.8)
    plt.show()

    return fila, columna


# --- Prueba de la función pistola() ---
posicion_disparo_demo = pistola()
print("Posición del disparo (demo):", posicion_disparo_demo)

## 6. Validación de impacto (WINNER / GAME OVER)

Comparamos la posición del pato (`pato()`) contra la del disparo (`pistola()`). Si coinciden → **acierto**: splash de sangre sobre el pato + pantalla `ganador.jpeg` con texto **WINNER** + sonido agudo. Si no coinciden → **fallo**: pantalla `gameover.jpg` con texto **GAME OVER** en rojo + sonido grave. En ambos casos se registra el resultado del disparo para las estadísticas finales (Sección 8).

> Recordatorio del flujo acordado: esto ocurre **en cada uno de los `NUM_DISPAROS` disparos** — el juego no se corta en el primer fallo, siempre se completan todos los disparos configurados. El resumen agregado va aparte, al final (Sección 7).

### 6.1 Comparar posiciones

Cada posición es una tupla `(fila, columna)`. En Python, comparar tuplas con `==` compara elemento por elemento automáticamente: `(2, 3) == (2, 3)` → `True`, `(2, 3) == (2, 1)` → `False`. No hace falta comparar `fila` y `columna` por separado.

### 6.2 Sonido sintetizado con NumPy (explicación con ejemplo)

Un tono puro es una onda senoidal: `y(t) = A·sin(2π·f·t)`, donde `f` es la frecuencia en Hz (qué tan agudo/grave suena) y `A` la amplitud (volumen). Generamos `t` como un arreglo de instantes de tiempo con `np.linspace`, y evaluamos la fórmula sobre **todo el arreglo a la vez**.

**Ejemplo:** para un tono de `f=440` Hz (nota La) y duración `0.01` s a `44100` muestras/segundo, el primer instante distinto de cero es `t=1/44100≈0.0000227` s, y `y(t) = sin(2π·440·0.0000227) ≈ sin(0.0628) ≈ 0.0627` — un valor cercano a 0 porque la onda recién empieza a subir.

También aplicamos una **envolvente** (fade-in/fade-out): multiplicamos el inicio y el final de la onda por una rampa de 0 a 1, para que el sonido no empiece ni termine de golpe (evita un "click" audible).

- **Acierto:** dos tonos agudos ascendentes (880 Hz → 1320 Hz), como un "ding" de éxito.
- **Fallo:** un tono grave y más largo (150 Hz), como un "buzz" de error.

### 6.3 Splash de sangre (manipulación de imágenes + aleatoriedad)

Generamos varias manchas circulares rojas en posiciones y radios aleatorios sobre una textura transparente, usando la misma idea de "distancia euclidiana ≤ radio" que ya usamos para dibujar la mira — pero esta vez pintando píxeles en vez de trazar una línea. Esa textura se combina con el tablero usando la **misma función `fusionar_sprite_en_fondo()`** de la Sección 4 (mezcla alfa) — no necesitamos escribir composición de imágenes de nuevo, la reutilizamos.

In [ ]:
# ============================================================
# BLOQUE A — SONIDO SINTETIZADO CON NUMPY
# ============================================================
TASA_MUESTREO = 44100  # muestras por segundo (calidad estándar de audio)


def generar_tono(frecuencia, duracion=0.3, amplitud=0.5, tasa_muestreo=TASA_MUESTREO):
    """
    Genera un tono puro (onda senoidal) de 'frecuencia' Hz y 'duracion' segundos.
    Aplica una envolvente lineal de fade-in/fade-out (10% inicial y final) para evitar clics.
    """
    t = np.linspace(0, duracion, int(tasa_muestreo * duracion), endpoint=False)
    onda = amplitud * np.sin(2 * np.pi * frecuencia * t)

    n_fade = max(1, int(0.1 * len(t)))
    envolvente = np.ones_like(onda)
    envolvente[:n_fade] = np.linspace(0, 1, n_fade)
    envolvente[-n_fade:] = np.linspace(1, 0, n_fade)

    return onda * envolvente


def sonido_acierto():
    """Reproduce dos tonos agudos ascendentes (880 Hz -> 1320 Hz), como un 'ding' de éxito."""
    onda = np.concatenate([generar_tono(880, 0.12), generar_tono(1320, 0.18)])
    display(Audio(onda, rate=TASA_MUESTREO, autoplay=True))


def sonido_fallo():
    """Reproduce un tono grave y prolongado (150 Hz), como un 'buzz' de error."""
    onda = generar_tono(150, 0.4)
    display(Audio(onda, rate=TASA_MUESTREO, autoplay=True))

In [ ]:
# ============================================================
# BLOQUE B — SPLASH DE SANGRE (manipulación de imágenes + aleatoriedad)
# ============================================================
def generar_mascara_splash(tam_px, n_manchas=6, color=(200, 0, 0)):
    """
    Genera una textura RGBA (tam_px x tam_px) con manchas circulares rojas en posiciones
    y radios aleatorios (NumPy), sobre fondo transparente. Simula un splash de impacto.
    """
    splash = np.zeros((tam_px, tam_px, 4), dtype=np.uint8)
    filas_px, columnas_px = np.mgrid[0:tam_px, 0:tam_px]

    centros_x = np.random.randint(0, tam_px, n_manchas)
    centros_y = np.random.randint(0, tam_px, n_manchas)
    radios = np.random.randint(tam_px // 10, tam_px // 4, n_manchas)

    for cx, cy, radio in zip(centros_x, centros_y, radios):
        distancia = np.sqrt((columnas_px - cx) ** 2 + (filas_px - cy) ** 2)
        mascara_mancha = distancia <= radio
        splash[mascara_mancha, 0] = color[0]
        splash[mascara_mancha, 1] = color[1]
        splash[mascara_mancha, 2] = color[2]
        splash[mascara_mancha, 3] = 180  # semi-transparente

    return splash

In [ ]:
# ============================================================
# BLOQUE C — PANTALLA DE REACCIÓN A PANTALLA COMPLETA (WINNER / GAME OVER)
# ============================================================
import matplotlib.patheffects as path_effects  # contorno del texto, para que se lea sobre cualquier fondo


def mostrar_pantalla_reaccion(ruta_imagen, texto, color_texto):
    """Muestra 'ruta_imagen' a pantalla completa con 'texto' grande centrado (con contorno negro)."""
    imagen = np.array(Image.open(ruta_imagen).convert("RGB"))
    alto, ancho = imagen.shape[0], imagen.shape[1]

    fig, ax = plt.subplots(figsize=(6, 6 * alto / ancho))
    ax.imshow(imagen)
    ax.set_xticks([])
    ax.set_yticks([])

    texto_dibujado = ax.text(
        ancho / 2, alto / 2, texto,
        color=color_texto, fontsize=42, weight="bold",
        ha="center", va="center",
    )
    texto_dibujado.set_path_effects([
        path_effects.Stroke(linewidth=4, foreground="black"),
        path_effects.Normal(),
    ])

    plt.show()

In [ ]:
# ============================================================
# BLOQUE D — VALIDACIÓN DE IMPACTO Y ORQUESTACIÓN DE LA REACCIÓN
# ============================================================
def validar_impacto(posicion_pato, posicion_disparo):
    """Compara la posición del pato y la del disparo. True si coinciden (acierto)."""
    return posicion_pato == posicion_disparo


def procesar_disparo(numero_disparo, posicion_pato, posicion_disparo):
    """
    Ejecuta la reacción visual y sonora del resultado (acierto o fallo) y
    devuelve un registro (dict) con los datos del disparo para las estadísticas.
    """
    acierto = validar_impacto(posicion_pato, posicion_disparo)

    if acierto:
        fila, columna = posicion_pato
        x_centro, y_centro = centro_celda(fila, columna, BORDES_FILA, BORDES_COLUMNA)
        splash = generar_mascara_splash(TAM_SPRITE)
        tablero_con_impacto = fusionar_sprite_en_fondo(FONDO_JUEGO, splash, x_centro, y_centro)

        fig, ax = plt.subplots(figsize=(6, 6 * ALTO_IMG / ANCHO_IMG))
        dibujar_tablero(ax, tablero_con_impacto, TAM_GRID, TAM_GRID, f"¡Impacto! -> celda ({fila},{columna})")
        plt.show()

        mostrar_pantalla_reaccion(RUTA_GANADOR, "WINNER", "lime")
        sonido_acierto()
    else:
        mostrar_pantalla_reaccion(RUTA_GAMEOVER, "GAME OVER", "red")
        sonido_fallo()

    return {
        "disparo_num": numero_disparo,
        "jugador": NOMBRE_JUGADOR,
        "pato_fila": posicion_pato[0],
        "pato_columna": posicion_pato[1],
        "disparo_fila": posicion_disparo[0],
        "disparo_columna": posicion_disparo[1],
        "acierto": acierto,
    }


# --- Prueba de procesar_disparo(): forzamos un acierto y un fallo para ver ambos casos ---
registro_demo_acierto = procesar_disparo(1, posicion_pato=(1, 1), posicion_disparo=(1, 1))
registro_demo_fallo = procesar_disparo(2, posicion_pato=(0, 0), posicion_disparo=(2, 2))
print(registro_demo_acierto)
print(registro_demo_fallo)

## 7. Bucle principal del juego

Repetimos el ciclo `pato()` → `pistola()` → `procesar_disparo()` exactamente `NUM_DISPAROS` veces (el valor configurado en la pantalla NES, Sección 2), sin cortar el juego aunque el jugador falle — tal como se definió: **todos** los disparos se juegan siempre, y cada uno queda registrado para las estadísticas finales.

Cada registro (un dict por disparo) se guarda en una lista; al terminar la partida, esa lista se convierte en una tabla de pandas en la Sección 8.

> ⚠️ **Nota:** con el `NUM_DISPAROS` configurado (por defecto 20), esta celda genera **muchas figuras** (tablero del pato + tablero del disparo + reacción, por cada disparo) y reproduce un sonido por disparo. Es intencional — el enunciado pide visualizar cada paso — pero puede tardar unos segundos en ejecutarse por completo.

In [ ]:
# ============================================================
# BLOQUE A — BUCLE PRINCIPAL DEL JUEGO
# ============================================================
def jugar_partida():
    """
    Ejecuta la partida completa: por cada uno de los NUM_DISPAROS disparos configurados,
    hace aparecer un pato, dispara, valida el impacto y registra el resultado.
    Devuelve la lista de registros (un dict por disparo).
    """
    registros = []
    for numero_disparo in range(1, NUM_DISPAROS + 1):
        print(f"\n=== Disparo {numero_disparo}/{NUM_DISPAROS} ===")
        posicion_pato = pato()
        posicion_disparo = pistola()
        registro = procesar_disparo(numero_disparo, posicion_pato, posicion_disparo)
        registros.append(registro)
    return registros


# ============================================================
# BLOQUE B — ORQUESTACIÓN: jugar la partida completa
# ============================================================
if NUM_DISPAROS is None or TAM_GRID is None:
    raise RuntimeError(
        "Falta configurar la partida. Vuelve a la Sección 2, usa la pantalla NES "
        "y presiona START antes de ejecutar esta celda."
    )

REGISTROS_PARTIDA = jugar_partida()
print(f"\nPartida terminada: {len(REGISTROS_PARTIDA)} disparos registrados para {NOMBRE_JUGADOR}.")

## 8. Pantalla final, estadísticas y registro en CSV

Con `REGISTROS_PARTIDA` (lista de dicts, uno por disparo) armamos un **DataFrame de pandas**, calculamos el resumen (`total`, `exitosos`, `errados`, `% aciertos`, `% fallos`) y mostramos la **pantalla final** reutilizando `gameover.jpg` — esta vez con el resumen de la partida superpuesto, para distinguirla claramente de la reacción de "fallo" de cada disparo individual (Sección 6).

Finalmente, agregamos una fila al **CSV histórico** (`registros_jugadores.csv`). Como se abre en modo `"a"` (append), cada partida de cada jugador se acumula sin borrar las anteriores — así se arma el registro multi-jugador para la Sección 9.

In [ ]:
# ============================================================
# BLOQUE A — ESTADÍSTICAS DE LA PARTIDA (pandas)
# ============================================================
def calcular_estadisticas(df_partida):
    """Calcula el resumen estadístico (total, exitosos, errados, %) a partir del DataFrame de disparos."""
    total = len(df_partida)
    exitosos = int(df_partida["acierto"].sum())
    errados = total - exitosos
    porcentaje_aciertos = round(100 * exitosos / total, 1) if total else 0.0
    porcentaje_fallos = round(100 * errados / total, 1) if total else 0.0

    return {
        "jugador": NOMBRE_JUGADOR,
        "grid": f"{TAM_GRID}x{TAM_GRID}",
        "total_disparos": total,
        "disparos_exitosos": exitosos,
        "disparos_errados": errados,
        "porcentaje_aciertos": porcentaje_aciertos,
        "porcentaje_fallos": porcentaje_fallos,
    }


# ============================================================
# BLOQUE B — PANTALLA FINAL (reutiliza gameover.jpg, resumen superpuesto)
# ============================================================
def mostrar_pantalla_final(estadisticas):
    """Muestra la pantalla de fin de partida (gameover.jpg) con el resumen estadístico superpuesto."""
    imagen = np.array(Image.open(RUTA_GAMEOVER).convert("RGB"))
    alto, ancho = imagen.shape[0], imagen.shape[1]

    fig, ax = plt.subplots(figsize=(6, 6 * alto / ancho))
    ax.imshow(imagen)
    ax.set_xticks([])
    ax.set_yticks([])

    titulo = ax.text(ancho / 2, alto * 0.20, "FIN DE LA PARTIDA", color="white",
                      fontsize=24, weight="bold", ha="center", va="center")
    titulo.set_path_effects([path_effects.Stroke(linewidth=4, foreground="black"), path_effects.Normal()])

    texto_stats = (
        f"Jugador: {estadisticas['jugador']}\n"
        f"Grid: {estadisticas['grid']}\n"
        f"Disparos: {estadisticas['total_disparos']}\n"
        f"Aciertos: {estadisticas['disparos_exitosos']} ({estadisticas['porcentaje_aciertos']}%)\n"
        f"Fallos: {estadisticas['disparos_errados']} ({estadisticas['porcentaje_fallos']}%)"
    )
    cuerpo = ax.text(ancho / 2, alto * 0.62, texto_stats, color="yellow",
                      fontsize=13, weight="bold", ha="center", va="center", linespacing=1.8)
    cuerpo.set_path_effects([path_effects.Stroke(linewidth=3, foreground="black"), path_effects.Normal()])

    plt.show()


# ============================================================
# BLOQUE C — REGISTRO EN CSV HISTÓRICO
# ============================================================
def guardar_registro_csv(estadisticas, ruta_csv=RUTA_CSV_REGISTROS):
    """Agrega una fila con las estadísticas de esta partida al CSV histórico (lo crea si no existe)."""
    fila = pd.DataFrame([{**estadisticas, "fecha_hora": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}])
    escribir_encabezado = not ruta_csv.exists()
    fila.to_csv(ruta_csv, mode="a", header=escribir_encabezado, index=False, encoding="utf-8")
    print(f"✅ Registro guardado en '{ruta_csv}' (partida de {estadisticas['jugador']}).")

In [ ]:
# ============================================================
# BLOQUE D — ORQUESTACIÓN: armar la tabla, mostrar resumen y guardar
# ============================================================
if not REGISTROS_PARTIDA:
    raise RuntimeError(
        "REGISTROS_PARTIDA está vacío. Vuelve a la Sección 7 y ejecuta jugar_partida() "
        "antes de continuar."
    )

DF_PARTIDA = pd.DataFrame(REGISTROS_PARTIDA)
ESTADISTICAS_PARTIDA = calcular_estadisticas(DF_PARTIDA)

print("Tabla de disparos de la partida:")
display(DF_PARTIDA)

print("\nResumen estadístico:")
display(pd.DataFrame([ESTADISTICAS_PARTIDA]))

mostrar_pantalla_final(ESTADISTICAS_PARTIDA)
guardar_registro_csv(ESTADISTICAS_PARTIDA)

## 9. Visualización estadística final

Cerramos el proyecto con los gráficos que pide el enunciado — **barras, histograma, pie chart y heatmap** — usando `pandas` + `Matplotlib` + `Seaborn`, y una tabla comparativa (*leaderboard*) armada a partir del CSV histórico acumulado entre partidas.

- **Barras:** cantidad de aciertos vs. fallos de esta partida.
- **Histograma:** distribución de la *distancia* entre el pato y el disparo en cada intento (cuánto se falló, no solo si se falló).
- **Pie chart:** proporción de aciertos/fallos.
- **Heatmap:** en qué celdas del grid apareció el pato con más frecuencia — reutiliza la matriz `TAM_GRID x TAM_GRID` como estructura de conteo.
- **Leaderboard:** todas las partidas registradas en `registros_jugadores.csv` (de todos los jugadores), ordenadas por % de aciertos.

### Distancia pato-disparo (explicación con ejemplo)

Para el histograma medimos qué tan lejos cayó cada disparo del pato, con distancia euclidiana entre celdas:

```
distancia = √((fila_pato - fila_disparo)² + (columna_pato - columna_disparo)²)
```

**Ejemplo:** pato en `(0, 0)`, disparo en `(1, 2)` → `distancia = √((0-1)² + (0-2)²) = √(1+4) = √5 ≈ 2.24`. Un acierto siempre da `distancia = 0`; cuanto más grande el número, más lejos estuvo el disparo del pato.

In [ ]:
# ============================================================
# BLOQUE A — PREPARACIÓN DE DATOS PARA LOS GRÁFICOS
# ============================================================
sns.set_theme(style="whitegrid")  # estilo consistente para todos los gráficos de Seaborn/Matplotlib


def agregar_columna_distancia(df_partida):
    """Agrega al DataFrame la distancia euclidiana entre la celda del pato y la del disparo."""
    df = df_partida.copy()
    df["distancia"] = np.sqrt(
        (df["pato_fila"] - df["disparo_fila"]) ** 2
        + (df["pato_columna"] - df["disparo_columna"]) ** 2
    )
    return df


def construir_matriz_frecuencia_pato(df_partida, tam_grid):
    """Cuenta cuántas veces apareció el pato en cada celda (fila, columna) del grid."""
    matriz = np.zeros((tam_grid, tam_grid), dtype=int)
    np.add.at(matriz, (df_partida["pato_fila"], df_partida["pato_columna"]), 1)
    return matriz


DF_PARTIDA = agregar_columna_distancia(DF_PARTIDA)
MATRIZ_FRECUENCIA_PATO = construir_matriz_frecuencia_pato(DF_PARTIDA, TAM_GRID)

In [ ]:
# ============================================================
# BLOQUE B — LOS 4 GRÁFICOS REQUERIDOS (barras, histograma, pie, heatmap)
# ============================================================
def graficar_barras_resultado(ax, df_partida):
    """Gráfico de barras: cantidad de disparos exitosos vs. errados."""
    conteo = df_partida["acierto"].map({True: "Acierto", False: "Fallo"}).value_counts()
    sns.barplot(x=conteo.index, y=conteo.values, hue=conteo.index,
                palette={"Acierto": "seagreen", "Fallo": "indianred"}, legend=False, ax=ax)
    ax.set_title("Aciertos vs. fallos")
    ax.set_xlabel("")
    ax.set_ylabel("Cantidad de disparos")


def graficar_histograma_distancia(ax, df_partida):
    """Histograma: distribución de la distancia entre el pato y el disparo."""
    sns.histplot(df_partida["distancia"], bins=8, kde=False, color="steelblue", ax=ax)
    ax.set_title("Distribución de la distancia pato-disparo")
    ax.set_xlabel("Distancia (celdas)")
    ax.set_ylabel("Frecuencia")


def graficar_pie_resultado(ax, estadisticas):
    """Pie chart: proporción de aciertos vs. fallos."""
    valores = [estadisticas["disparos_exitosos"], estadisticas["disparos_errados"]]
    etiquetas = [f"Aciertos ({estadisticas['porcentaje_aciertos']}%)",
                 f"Fallos ({estadisticas['porcentaje_fallos']}%)"]
    ax.pie(valores, labels=etiquetas, colors=["seagreen", "indianred"],
           autopct=lambda p: f"{p:.0f}%" if p > 0 else "", startangle=90)
    ax.set_title("Proporción de aciertos")


def graficar_heatmap_posiciones(ax, matriz_frecuencia):
    """Heatmap: frecuencia con la que el pato apareció en cada celda del grid."""
    sns.heatmap(matriz_frecuencia, annot=True, fmt="d", cmap="Reds", cbar=True, ax=ax)
    ax.set_title("Frecuencia de aparición del pato por celda")
    ax.set_xlabel("Columna")
    ax.set_ylabel("Fila")


fig, ejes = plt.subplots(2, 2, figsize=(12, 10))
graficar_barras_resultado(ejes[0, 0], DF_PARTIDA)
graficar_histograma_distancia(ejes[0, 1], DF_PARTIDA)
graficar_pie_resultado(ejes[1, 0], ESTADISTICAS_PARTIDA)
graficar_heatmap_posiciones(ejes[1, 1], MATRIZ_FRECUENCIA_PATO)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BLOQUE C — LEADERBOARD HISTÓRICO (todas las partidas registradas en el CSV)
# ============================================================
def cargar_historial(ruta_csv=RUTA_CSV_REGISTROS):
    """Carga el CSV histórico de partidas. Si todavía no existe, devuelve un DataFrame vacío."""
    if not ruta_csv.exists():
        return pd.DataFrame()
    return pd.read_csv(ruta_csv)


def mostrar_leaderboard(historial):
    """Muestra la tabla de partidas ordenada por % de aciertos, y un gráfico de barras comparativo."""
    if historial.empty:
        print("Todavía no hay partidas registradas en el historial.")
        return

    leaderboard = historial.sort_values("porcentaje_aciertos", ascending=False).reset_index(drop=True)
    print("Historial de partidas (todas las jugadoras y jugadores registrados):")
    display(leaderboard)

    etiquetas = leaderboard["jugador"] + " (" + leaderboard["fecha_hora"].str.slice(0, 16) + ")"
    fig, ax = plt.subplots(figsize=(9, max(3, 0.5 * len(leaderboard))))
    sns.barplot(x=leaderboard["porcentaje_aciertos"], y=etiquetas, hue=etiquetas,
                palette="viridis", legend=False, ax=ax)
    ax.set_title("Leaderboard — % de aciertos por partida")
    ax.set_xlabel("% de aciertos")
    ax.set_ylabel("")
    ax.set_xlim(0, 100)
    plt.tight_layout()
    plt.show()


HISTORIAL_PARTIDAS = cargar_historial()
mostrar_leaderboard(HISTORIAL_PARTIDAS)